In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import os
import shutil
import random
import numpy as np
from sklearn.metrics import classification_report

In [ ]:
# Function to split dataset into train and test directories
def create_train_test_dirs(data_path, train_ratio=0.8):
    categories = ['Broken soybeans', 'Immature soybeans', 'Intact soybeans',
                  'Skin-damaged soybeans', 'Spotted soybeans']

    for split in ['train', 'test']:
        for category in categories:
            os.makedirs(os.path.join(data_path, split, category), exist_ok=True)

    for category in categories:
        source_dir = os.path.join(data_path, category)
        if not os.path.exists(source_dir):
            print(f"Warning: Source directory not found: {source_dir}. Skipping this category.")
            continue

        files = os.listdir(source_dir)
        random.shuffle(files)
        train_count = int(len(files) * train_ratio)

        for i, file in enumerate(files):
            source_file = os.path.join(source_dir, file)
            dest_dir = os.path.join(data_path, 'train' if i < train_count else 'test', category)
            shutil.move(source_file, os.path.join(dest_dir, file))

In [ ]:
# Data augmentation and dataset loading
def load_dataset(dataset_path, batch_size=32, img_size=(224, 224)):
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=30,
        width_shift_range=0.3,
        height_shift_range=0.3,
        shear_range=0.3,
        zoom_range=0.3,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    test_datagen = ImageDataGenerator(rescale=1./255)

    train_dataset = train_datagen.flow_from_directory(
        os.path.join(dataset_path, 'train'),
        batch_size=batch_size,
        shuffle=True,
        target_size=img_size,
        class_mode='categorical'
    )

    test_dataset = test_datagen.flow_from_directory(
        os.path.join(dataset_path, 'test'),
        batch_size=batch_size,
        shuffle=False,
        target_size=img_size,
        class_mode='categorical'
    )

    return train_dataset, test_dataset


In [ ]:
# Building model with InceptionV3 and fine-tuning
def build_inception_model(input_shape=(224, 224, 3), num_classes=5):
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # Freeze base model initially

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model


In [ ]:
# Fine-tuning the model
def fine_tune_model(model, train_dataset, test_dataset):
    model.layers[0].trainable = True  # Unfreeze the base model

    for layer in model.layers[0].layers[:250]:
        layer.trainable = False  # Freeze initial layers

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
    ]

    model.fit(train_dataset, validation_data=test_dataset, epochs=20, callbacks=callbacks)


In [ ]:
# Path to your dataset
data_path = '/content/drive/MyDrive/soybean'

In [ ]:
# Create train/test directories
create_train_test_dirs(data_path)

In [ ]:
# Load dataset
train_dataset, test_dataset = load_dataset(data_path)

Found 4408 images belonging to 5 classes.
Found 1105 images belonging to 5 classes.


In [ ]:
# Build model
model = build_inception_model()

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
# Train the model (initially freeze InceptionV3 layers)
model.fit(train_dataset, validation_data=test_dataset, epochs=10)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 4260s 31s/step - accuracy: 0.3979 - loss: 1.7372 - val_accuracy: 0.6624 - val_loss: 0.8612
Epoch 2/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 73s 528ms/step - accuracy: 0.5593 - loss: 1.0713 - val_accuracy: 0.6878 - val_loss: 0.8155
Epoch 3/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 73s 526ms/step - accuracy: 0.5826 - loss: 1.0271 - val_accuracy: 0.6878 - val_loss: 0.7949
Epoch 4/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 89s 582ms/step - accuracy: 0.6033 - loss: 1.0048 - val_accuracy: 0.6950 - val_loss: 0.7684
Epoch 5/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 78s 567ms/step - accuracy: 0.6108 - loss: 0.9797 - val_accuracy: 0.6615 - val_loss: 0.8382
Epoch 6/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 73s 532ms/step - accuracy: 0.6136 - loss: 0.9602 - val_accuracy: 0.6824 - val_loss: 0.7995
Epoch 7/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 73s 533ms/step - accuracy: 0.6190 - loss: 0.9350 - val_accuracy: 0.7032 - val_loss: 0.7330
Epoch 8/10
138/138 ━━━━━━━━━━━━━━━━━━━━ 74s 536ms/step - accuracy: 0.6301 - loss: 0

In [ ]:
# Fine-tune the model
fine_tune_model(model, train_dataset, test_dataset)

Epoch 1/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 121s 686ms/step - accuracy: 0.6066 - loss: 0.9829 - val_accuracy: 0.7430 - val_loss: 0.6448 - learning_rate: 1.0000e-04
Epoch 2/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 75s 546ms/step - accuracy: 0.7427 - loss: 0.6965 - val_accuracy: 0.8027 - val_loss: 0.5675 - learning_rate: 1.0000e-04
Epoch 3/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 81s 543ms/step - accuracy: 0.7696 - loss: 0.6057 - val_accuracy: 0.8100 - val_loss: 0.5283 - learning_rate: 1.0000e-04
Epoch 4/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 75s 545ms/step - accuracy: 0.7951 - loss: 0.5434 - val_accuracy: 0.7937 - val_loss: 0.5707 - learning_rate: 1.0000e-04
Epoch 5/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 76s 547ms/step - accuracy: 0.8086 - loss: 0.5022 - val_accuracy: 0.8127 - val_loss: 0.5464 - learning_rate: 1.0000e-04
Epoch 6/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 74s 540ms/step - accuracy: 0.7998 - loss: 0.5260 - val_accuracy: 0.8208 - val_loss: 0.4979 - learning_rate: 1.0000e-04
Epoch 7/20
138/138 ━━━━━━━━━━━━━━━━━━━━ 74s 5

In [ ]:
# Evaluate model
test_images, test_labels = next(iter(test_dataset))  # Extract test images and labels
predictions = model.predict(test_dataset)
rounded_preds = np.argmax(predictions, axis=1)
true_labels = test_dataset.classes  # Correct way to get true labels

35/35 ━━━━━━━━━━━━━━━━━━━━ 16s 314ms/step


In [ ]:
print(classification_report(true_labels, rounded_preds, target_names=test_dataset.class_indices.keys()))


                       precision    recall  f1-score   support

      Broken soybeans       0.89      0.77      0.82       201
    Immature soybeans       0.79      0.76      0.77       225
      Intact soybeans       0.81      0.81      0.81       241
Skin-damaged soybeans       0.84      0.96      0.90       226
     Spotted soybeans       0.86      0.90      0.88       212

             accuracy                           0.84      1105
            macro avg       0.84      0.84      0.84      1105
         weighted avg       0.84      0.84      0.84      1105

